# RunwayMC — Parallel Computing & Architecture (PCAP) Project Demo
## Robust Aircraft Runway Scheduling Under Uncertainty
### **Parallel Implementations: Serial vs. OpenMP (Multi-Core CPU) vs. CUDA (NVIDIA GPU)**

---

### Project Overview & Parallel Computing Concepts
1. **The Problem:** The Aircraft Landing & Takeoff Scheduling Problem is NP-hard. Real-world operations suffer from stochastic flight delays, wake turbulence separation constraints, and unexpected runway closures.
2. **The Algorithm:** Simulated Annealing (SA) over candidate permutations evaluated across $S$ Monte Carlo weather/delay scenarios.
3. **PCAP Parallelization Strategies:**
   * **Serial Reference:** 1 thread executing $C$ sequential Markov chains.
   * **OpenMP Coarse-Grained (`omp_chains`):** Independent Markov chains partitioned across multi-core CPU threads (`#pragma omp parallel for schedule(dynamic)`).
   * **OpenMP Fine-Grained (`omp_scenarios`):** Parallel reduction over scenario evaluations across CPU threads.
   * **CUDA GPU Acceleration (`cuda`):** Massively parallel GPU kernels where CUDA thread blocks execute chains and warps evaluate scenarios using warp-shuffle reductions and shared memory.

> **Academic Disclaimer:** *This is an academic simulation with simplified separation standards. It is not validated for, and must not be used in, real air-traffic operations.*

## 1. Hardware & Environment Verification
Confirm that Google Colab has allocated an NVIDIA GPU (e.g. Tesla T4) with the CUDA toolkit.

In [ ]:
# Verify GPU device name, driver, memory, and CUDA compiler version
!nvidia-smi
!nvcc --version
!lscpu | grep 'Model name\|CPU(s):\|Thread'

## 2. Clone Repository & Setup Working Directory

In [ ]:
import os

# Clone from GitHub if not already present; otherwise pull latest
if not os.path.exists('/content/PCAP'):
    !git clone https://github.com/manoharadimalla/PCAP.git /content/PCAP
else:
    %cd /content/PCAP
    !git pull origin main

# Enter repository directory
%cd /content/PCAP
!git status
!python3 bench/env_info.py

## 3. Build Unified C++ Engine with CUDA & OpenMP

In [ ]:
# Configure with CMake (enabling CUDA and OpenMP)
!cmake -B build -DCMAKE_BUILD_TYPE=Release -DUSE_CUDA=ON -DUSE_OPENMP=ON

# Compile the binaries using all available CPU cores
!cmake --build build --config Release -j$(nproc)

# Check built binary info to confirm CUDA detection
!./build/runway info

## 4. Run Test Suite (Catch2 Verification)
Validates decoder correctness, runway closure handling, separation matrix triangle inequality, and OpenMP/CUDA equivalence.

In [ ]:
!ctest --test-dir build --output-on-failure

## 5. Interactive Demo: Schedule an Input Flight Batch
Here we provide an input aircraft batch (`data/instances/demo_interactive.csv` with 20 flights) and solve it using **Serial CPU**, **OpenMP Multi-Core CPU**, and **CUDA GPU**.

This demonstrates the actual practical output: delay reduction + execution speedup!

In [ ]:
import pandas as pd
import json

instance_file = "data/instances/demo_interactive.csv"

# 1. Display input flight data
print("=== INPUT AIRCRAFT FLIGHT BATCH ===")
df_in = pd.read_csv(instance_file)
display(df_in.head(10))
print(f"Total flights in this batch: {len(df_in)}")

# 2. Run Serial CPU Solve
print("\n[1/3] Running Serial CPU Solve...")
!./build/runway solve --instance {instance_file} --impl serial --chains 16 --scenarios 1024 --iters 1000 --seed 42 --out results/demo_serial.json

# 3. Run OpenMP Multi-Core CPU Solve
print("\n[2/3] Running OpenMP Multi-Core Solve...")
!./build/runway solve --instance {instance_file} --impl omp --omp-mode chains --threads 2 --chains 16 --scenarios 1024 --iters 1000 --seed 42 --out results/demo_omp.json

# 4. Run CUDA GPU Solve
print("\n[3/3] Running CUDA GPU Accelerated Solve...")
!./build/runway solve --instance {instance_file} --impl cuda --chains 16 --scenarios 1024 --iters 1000 --seed 42 --out results/demo_cuda.json

# 5. Compare Results & Speedups
with open('results/demo_serial.json') as f: res_ser = json.load(f)
with open('results/demo_omp.json') as f: res_omp = json.load(f)
with open('results/demo_cuda.json') as f: res_cuda = json.load(f)

t_ser = res_ser['timings']['kernel_ms']
t_omp = res_omp['timings']['kernel_ms']
t_cuda = res_cuda['timings']['kernel_ms']

fcfs_cost = res_ser['metrics']['fcfs']['mean_cost']
opt_cost = res_ser['metrics']['best']['mean_cost']
savings_pct = (fcfs_cost - opt_cost) / fcfs_cost * 100.0

summary = [
    {"Implementation": "Serial (1 Thread)", "Kernel Time (ms)": f"{t_ser:.2f}", "Speedup vs Serial": "1.00x", "Simulations/Sec": f"{res_ser['timings']['sim_per_s']:,.0f}"},
    {"Implementation": "OpenMP (Multi-Core CPU)", "Kernel Time (ms)": f"{t_omp:.2f}", "Speedup vs Serial": f"{t_ser / t_omp:.2f}x", "Simulations/Sec": f"{res_omp['timings']['sim_per_s']:,.0f}"},
    {"Implementation": "CUDA (NVIDIA GPU)", "Kernel Time (ms)": f"{t_cuda:.2f}", "Speedup vs Serial": f"{t_ser / t_cuda:.2f}x", "Simulations/Sec": f"{res_cuda['timings']['sim_per_s']:,.0f}"},
]

print("\n" + "="*65)
print("             PARALLEL PERFORMANCE COMPARISON (PCAP)")
print("="*65)
display(pd.DataFrame(summary))

print(f"\nSchedule Quality Improvement:")
print(f"  • Baseline FCFS Expected Delay Cost : {fcfs_cost:.1f} seconds")
print(f"  • RunwayMC Optimized Expected Delay : {opt_cost:.1f} seconds")
print(f"  • Total Delay Reduction Achieved   : {savings_pct:.1f}% reduction!")

## 6. Run Full PCAP Benchmark Suite (E1 – E5)
Executes all 5 experiments covering:
* **E1:** Unit tests & determinism gate.
* **E2:** Solution quality vs. Baselines (FCFS vs Random Search vs SA on 20,000 unseen test days).
* **E3:** OpenMP CPU scaling across thread counts (Chains vs Scenarios).
* **E4:** Throughput & Scaling across scenario counts $S \in [128, 4096]$: **Serial vs. OpenMP vs. CUDA**.
* **E5:** Value of robust stochastic planning vs. deterministic planning.

In [ ]:
# Run the full benchmark harness and generate publication-quality plots
!python3 bench/run_bench.py --all
!python3 scripts/make_plots.py

## 7. Display Generated Benchmark & Speedup Plots
These are the exact figures you will present in your PCAP evaluation.

In [ ]:
from IPython.display import Image, display
import glob

figures = sorted(glob.glob('results/figures/*.png'))
for fig_path in figures:
    print(f"\n=== {fig_path} ===")
    display(Image(filename=fig_path, width=700))